# Can we predict 30-day readmission?
Target: `readmit_30d`. Data: one row per hospital stay; the same person appears in many rows.

**Key design point:** people have ~8 stays each, so a random row-level split would put the same person in train and test and overstate performance. All evaluation here uses **grouped** cross-validation (on `person_key`), which mimics predicting for patients the model has never seen.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve, brier_score_loss
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance
pd.set_option('display.width', 140)
df = pd.read_csv('data.csv')
pd.read_csv('columns.csv')

## 1. Data checks

In [ ]:
print(df.shape, '| missing:', df.isna().sum().sum(), '| duplicate rows:', df.duplicated().sum())
print('persons:', df.person_key.nunique(), '| stays/person (min, median, max):', df.groupby('person_key').size().describe()[['min','50%','max']].tolist())
y = df.readmit_30d
p = y.mean()
print('readmit rate: %.3f' % p)
# are demographics constant within a person?
print(df.groupby('person_key')[['birth_year','sex','postcode_area']].nunique().max())
df.describe().T

## 2. Is there a person-level effect?
If readmission were independent across stays, the spread of per-person rates would match a binomial. Compare.

In [ ]:
g = df.groupby('person_key').readmit_30d.agg(['size','mean'])
exp_sd = np.sqrt(p*(1-p)/g['size']).mean()
print('observed SD of person rates: %.3f   expected if independent: ~%.3f' % (g['mean'].std(), exp_sd))
print('persons with zero readmissions: %.1f%%' % (100*(g['mean']==0).mean()))

## 3. Univariate relationships with readmission

In [ ]:
num = ['visit_number','birth_year','postcode_area','bmi','length_of_stay_days','n_medications','lab_a','lab_b']
fig, ax = plt.subplots(2, 4, figsize=(16, 7))
for a, c in zip(ax.ravel(), num):
    q = pd.qcut(df[c], 10, duplicates='drop')
    r = y.groupby(q, observed=True).mean()
    a.plot(range(len(r)), r.values, 'o-'); a.axhline(p, ls='--', c='gray')
    a.set_title(c); a.set_xlabel('decile'); a.set_ylabel('readmit rate')
plt.tight_layout(); plt.show()
print(df.groupby('sex').readmit_30d.mean())
print(df[num+['readmit_30d']].corr()['readmit_30d'].round(3))

## 4. Models (grouped, stratified 5-fold CV, out-of-fold predictions)

In [ ]:
feats = num + ['sex']
X = df[feats]
groups = df.person_key
cv = StratifiedGroupKFold(5, shuffle=True, random_state=0)
pre = ColumnTransformer([('n', StandardScaler(), num), ('c', OneHotEncoder(drop='if_binary'), ['sex'])])
X_gb = X.assign(sex=(X.sex == 'F').astype(int))
models = {
    'baseline (prevalence)': (DummyClassifier(strategy='prior'), X),
    'logistic regression': (make_pipeline(pre, LogisticRegression(max_iter=1000)), X),
    'gradient boosting': (HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, random_state=0), X_gb),
}
oof = {n: cross_val_predict(m, Xi, y, groups=groups, cv=cv, method='predict_proba')[:, 1] for n, (m, Xi) in models.items()}

idx = df.groupby('person_key').indices
ids = np.array(list(idx))
def boot(f, pr, B=300, seed=0):
    rng = np.random.default_rng(seed); v = []
    for _ in range(B):
        s = np.concatenate([idx[i] for i in rng.choice(ids, len(ids))])
        v.append(f(y.values[s], pr[s]))
    return np.percentile(v, [2.5, 97.5])
rows = []
for n, pr in oof.items():
    lo, hi = boot(roc_auc_score, pr)
    rows.append((n, roc_auc_score(y, pr), f'{lo:.3f}-{hi:.3f}', average_precision_score(y, pr), brier_score_loss(y, pr)))
res = pd.DataFrame(rows, columns=['model','ROC AUC','AUC 95% CI (person bootstrap)','PR AUC','Brier']).set_index('model').round(3)
print('prevalence (= PR AUC of a random model): %.3f' % p)
res

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
for n, pr in oof.items():
    if 'baseline' in n: continue
    f, t, _ = roc_curve(y, pr); ax[0].plot(f, t, label=n)
    pp, rr, _ = precision_recall_curve(y, pr); ax[1].plot(rr, pp, label=n)
    fp, mp = calibration_curve(y, pr, n_bins=10, strategy='quantile'); ax[2].plot(mp, fp, 'o-', label=n)
ax[0].plot([0,1],[0,1],'k--'); ax[0].set(title='ROC', xlabel='FPR', ylabel='TPR')
ax[1].axhline(p, c='k', ls='--'); ax[1].set(title='Precision-recall', xlabel='recall', ylabel='precision')
ax[2].plot([0,.5],[0,.5],'k--'); ax[2].set(title='Calibration', xlabel='predicted', ylabel='observed')
ax[0].legend(); plt.tight_layout(); plt.show()

## 5. What drives the signal?

In [ ]:
lr = models['logistic regression'][0].fit(X, y)
coef = pd.Series(lr[-1].coef_[0], index=num+['sex_M']).sort_values()
print('Standardised logistic coefficients (log-odds per 1 SD):'); print(coef.round(3))
gb = models['gradient boosting'][0].fit(X_gb, y)
pi = permutation_importance(gb, X_gb, y, scoring='roc_auc', n_repeats=5, random_state=0)
pd.Series(pi.importances_mean, index=feats).sort_values().plot.barh(title='Permutation importance (AUC drop; model fit on all data)')
plt.show()

## 6. Does the model help in practice? Lift by risk decile (out-of-fold, logistic regression)

In [ ]:
t = pd.DataFrame({'p': oof['logistic regression'], 'y': y})
t['decile'] = pd.qcut(t.p.rank(method='first'), 10, labels=range(1, 11))
lift = t.groupby('decile', observed=True).agg(n=('y','size'), mean_pred=('p','mean'), observed=('y','mean')).round(3)
lift['lift'] = (lift.observed / p).round(2)
print(lift)
top = t[t.decile == 10]
print('Flagging the top 10%% of risk captures %.1f%% of readmissions' % (100*top.y.sum()/t.y.sum()))

## 7. Caveats
- Evaluation is grouped by person, so numbers reflect performance on unseen patients.
- `person_key` is deliberately **not** a feature: it is an identifier and would not generalise to new patients.
- Associations only; no causal claims. There is no calendar-time variable, so temporal validation was not possible.
- `length_of_stay_days` and `n_medications` are known at discharge, so this is a discharge-time model, not an admission-time one.